# OneLake Security Error — Detect & Refresh

Detects OneLake security errors on Direct Lake semantic models and refreshes only the affected models to re-sync.

Edit the **Configuration** cell with your workspace and model names, then run all cells.

> **Requires:** XMLA read/write enabled on the capacity and Contributor role on the workspace.

In [ ]:
WORKSPACE = "Your-Workspace-Name"

MODELS = [
    "Semantic Model A",
    "Semantic Model B",
]

DAX_QUERY = "EVALUATE ROW(\"OK\", 1)"

In [ ]:
import sempy.fabric as fabric
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from notebookutils import mssparkutils

In [ ]:
ONELAKE_ERROR_PATTERNS = [
    "OneLake security configuration has changed",
    "transient issue when trying to determine user permissions defined in OneLake",
    "Universal security version mismatch error on artifact",
]

def _is_onelake_security_error(error_text: str) -> bool:
    lower = error_text.lower()
    return any(p.lower() in lower for p in ONELAKE_ERROR_PATTERNS)

def _probe_model(model_name: str) -> dict:
    try:
        fabric.evaluate_dax(dataset=model_name, workspace=WORKSPACE, dax_string=DAX_QUERY)
        return {"model": model_name, "status": "ok"}
    except Exception as e:
        error_text = str(e)
        if _is_onelake_security_error(error_text):
            return {"model": model_name, "status": "security_error", "error": error_text}
        return {"model": model_name, "status": "other_error", "error": error_text}

print(f"Probing {len(MODELS)} model(s) in '{WORKSPACE}'...\n")
results = []

with ThreadPoolExecutor(max_workers=len(MODELS)) as executor:
    futures = {executor.submit(_probe_model, m): m for m in MODELS}
    for future in as_completed(futures):
        r = future.result()
        results.append(r)
        icon = "✓" if r["status"] == "ok" else "✗" if r["status"] == "security_error" else "⚠"
        detail = "" if r["status"] == "ok" else f" — {r['error'][:120]}"
        print(f"  {icon} {r['model']}: {r['status']}{detail}")

errors_detected = [r for r in results if r["status"] == "security_error"]
print(f"\nOneLake security errors: {len(errors_detected)} of {len(MODELS)}")

In [ ]:
if not errors_detected:
    print("No OneLake security errors — nothing to refresh. ✓")
else:
    workspace_id = fabric.resolve_workspace_id(WORKSPACE)

    # Load .NET TOM assembly for XMLA
    _loader = fabric.create_tom_server(readonly=True, workspace=workspace_id)
    _loader.Dispose()
    import Microsoft.AnalysisServices.Tabular as TOM

    def _send_xmla_refresh(dataset_name: str, refresh_type: str):
        token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
        conn_str = (
            f"Provider=MSOLAP;"
            f"Data Source=powerbi://api.powerbi.com/v1.0/myorg/{WORKSPACE};"
            f"Password={token};"
        )
        tmsl = json.dumps({"refresh": {"type": refresh_type, "objects": [{"database": dataset_name}]}})
        server = TOM.Server()
        server.Connect(conn_str)
        try:
            result = server.Execute(tmsl)
            for i in range(result.Count):
                for j in range(result[i].Messages.Count):
                    msg = result[i].Messages[j]
                    if hasattr(msg, "Description") and msg.Description:
                        raise RuntimeError(msg.Description)
        finally:
            server.Disconnect()
            server.Dispose()

    def _refresh_model(model_name: str):
        print(f"\nRefreshing: {model_name}")

        automatic_ok = True
        start_ms = int(time.time() * 1000)
        try:
            _send_xmla_refresh(model_name, "automatic")
        except RuntimeError as e:
            print(f"  Automatic refresh error: {e}")
            automatic_ok = False

        need_full = not automatic_ok
        if automatic_ok:
            refreshes = fabric.list_refresh_requests(dataset=model_name, workspace=WORKSPACE)
            if refreshes is not None and len(refreshes) > 0:
                latest = refreshes.iloc[0]
                latest_start = (
                    int(latest["Start Time"].timestamp() * 1000)
                    if hasattr(latest["Start Time"], "timestamp")
                    else int(latest["Start Time"])
                )
                if latest_start > start_ms and latest["Status"] == "Completed":
                    print(f"  ✓ Automatic refresh verified")
                    need_full = False
                else:
                    need_full = True
            else:
                need_full = True

        if need_full:
            print(f"  Falling back to full refresh...")
            _send_xmla_refresh(model_name, "full")
            print(f"  ✓ Full refresh completed")

    models_to_refresh = [r["model"] for r in errors_detected]
    failed = []

    print(f"Refreshing {len(models_to_refresh)} affected model(s)...")

    with ThreadPoolExecutor(max_workers=len(models_to_refresh)) as executor:
        futures = {executor.submit(_refresh_model, m): m for m in models_to_refresh}
        for future in as_completed(futures):
            model = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"  ERROR refreshing {model}: {e}")
                failed.append(model)

    print(f"\nDone. {len(models_to_refresh) - len(failed)}/{len(models_to_refresh)} refreshed.")
    if failed:
        print(f"FAILED: {failed}")